## Sample Weights

This notebook will cover exercise answer.

* Exercise 4.6
* Exercise 4.7

As we go along, there will be some explanations.

More importantly, this method can be applied not just within mean-reversion strategy but also other strategies as well.

Most of the functions below can be found under research/Sampling.

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

dollar = pd.read_csv('../sample-data/dollar_bars.txt', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

In [ ]:
dollar = rs.bband_as_side(data = dollar, 
                          window = 50, 
                          width = 0.009)

dollar['volatility'] = rs.vol(dollar['close'], span0 = 50)

# volatility is a return; cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(dollar['close'], 
                    limit = dollar['volatility'].mean() * dollar['close'].mean())

vb = rs.vert_barrier(data = dollar['close'], 
                 events = events, 
                 period = 'days', 
                 freq = 1)

tb = rs.tri_barrier(data = dollar['close'], 
                    events = events, 
                    trgt = dollar['volatility'] * 5, 
                    min_req = 0.002, 
                    num_threads = 3, 
                    ptSl = [0,2],
                    t1 = vb, 
                    side = dollar['side'])

mlabel = rs.meta_label(data = dollar['close'], 
                       events = tb, 
                       drop = False) # when you have a side binary, you won't have rare labels usually

In [ ]:
tb

In [ ]:
#this is a modified func using mp_pandas_obj, run up to 700x faster depending on data size.

idxM0 = rs.mp_idx_matrix(data = dollar['close'], events = tb)

idxM0

### Sequential bootstrap vs Standard Bootstrap

**A short introduction to the methods**

For standard bootstrap, we will randomly choose columns given their size. As the law of large number goes, as number of sample increase beyond 30.

It will start to display normal-like distribution. So more loops equals to higher reliability.

For sequential bootstrap, we choose based on known uniquness, therefore we will likely experience higher overall uniqueness.

However when samples get large enough, there will be a gradual drop in uniqueness. 

Given updated probability, we are less likely to choose the same column twice.

As sequential bootstrap develop into normal-like distribution, it will start to "converge" to mean with samples that are more unique compared to standard bootstrap method.

Hence for sequential bootstrap to be reliable, another method call monte-carlos was introduced.

**Note**

Due to the lack of resource, the Monte-Carlos Sequential Bootstrap (MC_seq_bts) was implemented but was not used. It will take high computational power to run this algorithm.

In [ ]:
def unique_check(idxM0, iter_num):
    i = 0
    total = 0
    while i < (iter_num + 1):
        phi_ = np.random.choice(idxM0.columns, size = idxM0.shape[1])
        stdU = rs.av_unique(idxM0[phi_]).mean()
        if stdU > total:
            total = stdU
        i += 1
    print ("Highest Standard Uniqueness after {0} loops: {1}".format(iter_num, total))
    phi = rs.mp_seq_bts(idxM = idxM0, sample_len = iter_num)
    seqU = rs.av_unique(idxM0[phi]).mean()
    print ("Ave Sequential Uniqueness after {0} loops: {1}".format(iter_num, seqU))

In [ ]:
# As n < 30

unique_check(idxM0, 20)

In [ ]:
# As n > 30

unique_check(idxM0, 50)

**Note**

Due to resource constraint, we used func unique_check.

However, this is not a fair comparison since phi_ randomly choose all the columns with replacement.

As such, there won't be much difference in their uniqueness for standard method, hence we will use the highest unique sample based on standard bootstrap to compare against sequential bootstrap sample's mean uniqueness.

The idea however is to let you know the difference when we use bootstrap in sklearn vs if we were to implement a sequential bootstrap for sklearn classifier to perform sampling.

For sequential bootstrap, we will randomly choose columns but we use the previous columns choosen to compare (in-the-bag). In that way, only samples that are unique will be used to train ML models.

As a result, it will not report an inflated OOB score.

**Interesting facts**

In most cases given a large data sample, choosen sample's mean uniqueness can remain 1.0 (highest uniqueness) for 3 - 10 loops.

In [ ]:
# Exercise 4.6

def idx_matrix(data: pd.Series, events: pd.DataFrame):
    '''
    AFML pg 63 snippet 4.3
    Calculates the number of times events sample overlaps each other
    Some simple moification included
    
    logic is still based on initial func stated in AFML pg 63
    
    This func has been modified to fit the example given
       
    This version of idx_matrix will take up to 30 minutes when tested against (DataFrame.shape(4025, 1840))
    '''
    indM_ = pd.DataFrame(0, index = data, columns=np.arange(events.t1.shape[0]))
    for i,(t0,t1) in enumerate(events.t1.items()):
        indM_.loc[t0:t1,i] = 1.
    
    idxM = indM_[indM_.sum(axis = 1) != 0]
    return idxM

def seq_bts(idxM: pd.DataFrame, phi: list, sample_len = None):
    '''
    Modified to fit the example
    '''
    # Generate a sample via sequential bootstrap
    if sample_len is None:
        sample_len=idxM.shape[1]
    
    avgU=pd.Series(0., dtype=float)
    for i in idxM:
        indM_ = idxM[phi + [i]] # reduce indM
        avgU.loc[i] = rs.av_unique(indM_).iloc[-1] #get the last value, if it is very unique it's value will be high
    prob = avgU/ avgU.sum() # this part lets you know the uniqueness of the entire series based on it's pick, hence assign prob
    print(prob)
    phi.append(np.random.choice(idxM.columns, p=prob))
    return phi


t1 = pd.Series([2,3,5], index=[0,2,4], name = 't1')
bar = np.arange(t1.max() + 1)
event = pd.Series([2,3,5], index=[0,2,4], name = 't1').to_frame()
idxM1 = idx_matrix(data = bar, events = event)

idxM1

In [ ]:
#if you run this a few more times, but if your dataset is limited, it will still not achieve the goal.
phi = [1,2]
out = seq_bts(idxM = idxM1, sample_len = None, phi = phi )

print("\nCurrent samples: {0}\nNext pick would most likely be column {1}".format(out, out[-1]))

In [ ]:
# Exercise 4.7

phi = [1,2,2]
out = seq_bts(idxM = idxM1, sample_len = None, phi = phi )

#As mentioned above, if the same column was picked twice even if it is unique, it is unlikely to be picked twice.
print("\nCurrent samples: {0}\nNext pick would most likely be column {1}".format(out, out[-1]))

### Conclusion

As mentioned previously, the func will keep picking what it considered as unique.

If you notice, both exercise outcome favors column 0 as the next pick.

Because it is considered unique to the current list, since it was the only one that was not picked.

As mentioned earlier, sequential boostrap will reduce the probability of a sample being repick. 

But given a large dataset, its overall uniqueness will gradually drop since getting a highly unique column is not possible, if the most unique ones already exist within the sample group.

This is to maintain it's uniqueness of all samples choosen.

**Note**

Probability of choosing maybe high, but it is still random hence it may not be reflected if the func was to be run a few more times.